# NB10 — Reviewer 2: spectral normalization sensitivity under experiment shift

**Project:** `BANANA_RIPENESS_MULTISENSOR_Q1`

This notebook addresses the pre-submission chemometrics review regarding the possibility that the observed experiment shift is driven mainly by additive/multiplicative spectral effects.

## Pre-specified sensitivity analyses

The frozen raw analysis from NB03/NB04 is **not modified**. Two new preprocessing arms are evaluated:

1. **SNV** — Standard Normal Variate applied independently to each observation across the 18 AS7265x spectral bands.
2. **DOMAIN_Z** — unsupervised per-experiment z-score normalization. For each experiment, each feature is centered and scaled using the mean and standard deviation computed from that experiment's **unlabeled feature values only**.

`DOMAIN_Z` is explicitly a transductive diagnostic: it assumes a new deployment batch can accumulate unlabeled measurements before classification.

## Modalities

- `Spectral`: 18 AS7265x bands, 410–940 nm.
- `TPH_Spectral`: internal temperature, pressure, humidity + the 18 spectral bands.

## Validation

For each sensitivity arm:
- the original nested Leave-One-Experiment-Out protocol is retained;
- the original compact hyperparameter grid is retained;
- the original lexicographic hyperparameter-selection rule is retained;
- no outer labels are used in preprocessing, model fitting, or hyperparameter selection;
- all five model families are re-selected internally under the transformed representation.

The frozen raw NB04 results are imported only as the reference condition.

**Hardware:** CPU only.


In [ ]:
# 0. Environment and dependencies
from google.colab import drive
drive.mount('/content/drive')

import sys, subprocess, importlib.util

if importlib.util.find_spec('mord') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mord==0.7'])
if importlib.util.find_spec('xgboost') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, platform, shutil, time, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    matthews_corrcoef, cohen_kappa_score, mean_absolute_error
)

import mord
import xgboost as xgb

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

SEED = 2026
np.random.seed(SEED)

PROJECT = Path('/content/drive/MyDrive/Paper3_Banano/BANANA_RIPENESS_MULTISENSOR_Q1')
MASTER_PATH = PROJECT/'02_EXTRACTED_DATA'/'06_master_original_multisensor.csv'
PROTOCOL_PATH = PROJECT/'04_FROZEN_SPLITS'/'nested_loeo_protocol_v1.json'
RESULTS_ROOT = PROJECT/'06_RESULTS'

assert MASTER_PATH.exists(), MASTER_PATH
assert PROTOCOL_PATH.exists(), PROTOCOL_PATH
assert RESULTS_ROOT.exists(), RESULTS_ROOT

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = RESULTS_ROOT/f'NB10_REVIEWER2_SPECTRAL_NORMALIZATION_{RUN_ID}'
TABLE_DIR = RUN_DIR/'tables'
FIG_DIR = RUN_DIR/'figures'
TEXT_DIR = RUN_DIR/'text'
META_DIR = RUN_DIR/'metadata'
for d in [TABLE_DIR, FIG_DIR, TEXT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ALL_LABELS = [1,2,3,4,5]
OUTER_TESTS = [2,3,4,5]
METHODS = ['SNV','DOMAIN_Z']
MODALITY_NAMES = ['Spectral','TPH_Spectral']
MODELS = ['OrdinalLogistic','PLSDA','SVM_RBF','RandomForest','XGBoost']
N_JOBS = max(1, min(8, (os.cpu_count() or 2)-1))

start_all = time.perf_counter()

print('Project:', PROJECT)
print('Output:', RUN_DIR)
print('CPU workers:', N_JOBS)
print('Sensitivity methods:', METHODS)


In [ ]:
# 1. Frozen provenance and canonical data
def sha256_file(path, chunk_size=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(chunk_size),b''):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_MASTER_SHA = '0a06fb3773f85dacee42e18eecf17e5e2a8e7f5e9508cd3d60bd4b66757e8ea8'
EXPECTED_PROTOCOL_SHA = '86731ff9a292f1afd35a215105b14c4af3aa2bda5e8c4dad4df639d4d82ad930'

master_sha = sha256_file(MASTER_PATH)
protocol_sha = sha256_file(PROTOCOL_PATH)

assert master_sha == EXPECTED_MASTER_SHA, f'Master data hash drift: {master_sha}'
assert protocol_sha == EXPECTED_PROTOCOL_SHA, f'Protocol hash drift: {protocol_sha}'

with open(PROTOCOL_PATH,'r',encoding='utf-8') as f:
    protocol=json.load(f)

df=pd.read_csv(MASTER_PATH,low_memory=False)

if 'is_synthetic_inferred' in df.columns:
    syn=df['is_synthetic_inferred']
    if syn.dtype != bool:
        syn=syn.astype(str).str.lower().map({'true':True,'false':False}).fillna(False)
    df=df.loc[~syn].copy()

df=df.reset_index(drop=True)
df['test_id']=df['test_id'].astype(int)
df['stage']=df['stage'].astype(int)

assert len(df)==17662
assert df['master_id'].is_unique
assert sorted(df['test_id'].unique().tolist())==OUTER_TESTS
assert protocol['outer_tests']==OUTER_TESTS

source_hashes=pd.DataFrame([
    {'artifact':MASTER_PATH.name,'sha256':master_sha,'bytes':MASTER_PATH.stat().st_size},
    {'artifact':PROTOCOL_PATH.name,'sha256':protocol_sha,'bytes':PROTOCOL_PATH.stat().st_size},
])
source_hashes.to_csv(META_DIR/'source_hashes.csv',index=False)

print('Frozen provenance: PASS')
print('Original observations:',f'{len(df):,}')
display(pd.crosstab(df.test_id,df.stage).reindex(columns=ALL_LABELS,fill_value=0))


In [ ]:
# 2. Modalities and preprocessing definitions
TPH_FEATURES=['Temp-int','Press-int','Humid-int']
SPECTRAL_FEATURES=[
    'SpA410','SpB435','SpC460','SpD485','SpE510','SpF535',
    'SpG560','SpH585','SpR610','SpI645','SpS680','SpJ705',
    'SpT730','SpU760','SpV810','SpW860','SpK900','SpL940'
]
MODALITIES={
    'Spectral':SPECTRAL_FEATURES,
    'TPH_Spectral':TPH_FEATURES+SPECTRAL_FEATURES,
}

for name,features in MODALITIES.items():
    missing=[c for c in features if c not in df.columns]
    assert not missing,(name,missing)
    assert df[features].notna().all().all()

def apply_snv_matrix(X, feature_names):
    X=np.asarray(X,float).copy()
    spec_idx=[feature_names.index(c) for c in SPECTRAL_FEATURES if c in feature_names]
    S=X[:,spec_idx]
    row_mean=S.mean(axis=1,keepdims=True)
    row_sd=S.std(axis=1,ddof=1,keepdims=True)
    row_sd=np.where((~np.isfinite(row_sd)) | (row_sd<1e-12),1.0,row_sd)
    X[:,spec_idx]=(S-row_mean)/row_sd
    return X

def transformed_matrix(frame, modality, method):
    features=MODALITIES[modality]
    X=frame[features].to_numpy(float)

    if method=='RAW':
        return X

    if method=='SNV':
        return apply_snv_matrix(X,features)

    if method=='DOMAIN_Z':
        # Label-free transductive normalization:
        # statistics are computed independently within each experiment.
        out=np.empty_like(X,dtype=float)
        test_ids=frame['test_id'].to_numpy(int)
        for t in sorted(np.unique(test_ids)):
            idx=np.where(test_ids==t)[0]
            Xt=X[idx]
            mu=Xt.mean(axis=0,keepdims=True)
            sd=Xt.std(axis=0,ddof=0,keepdims=True)
            sd=np.where((~np.isfinite(sd)) | (sd<1e-12),1.0,sd)
            out[idx]=(Xt-mu)/sd
        return out

    raise ValueError(method)

TRANSFORMED={}
for method in ['RAW']+METHODS:
    for modality in MODALITY_NAMES:
        X=transformed_matrix(df,modality,method)
        assert X.shape==(len(df),len(MODALITIES[modality]))
        assert np.isfinite(X).all()
        TRANSFORMED[(method,modality)]=X

print('Precomputed transformed matrices:')
for k,v in TRANSFORMED.items():
    print(k,v.shape)


In [ ]:
# 3. Spectral duplicate audit within each experiment
dup_rows=[]
for t,g in df.groupby('test_id',sort=True):
    n=len(g)
    unique_raw=g[SPECTRAL_FEATURES].drop_duplicates().shape[0]

    Xsnv=TRANSFORMED[('SNV','Spectral')][g.index.to_numpy()]
    unique_snv=np.unique(np.round(Xsnv,12),axis=0).shape[0]

    dup_rows.append({
        'test_id':int(t),
        'n_rows':n,
        'unique_raw_spectral_vectors':int(unique_raw),
        'raw_duplicate_fraction':float(1-unique_raw/n),
        'unique_snv_vectors_rounded_1e12':int(unique_snv),
        'snv_duplicate_fraction_rounded_1e12':float(1-unique_snv/n),
    })

duplicate_audit=pd.DataFrame(dup_rows)
duplicate_audit.to_csv(TABLE_DIR/'01_spectral_duplicate_audit_by_experiment.csv',index=False)
display(duplicate_audit.round(4))


In [ ]:
# 4. PCA diagnostic: does PC1 dominance collapse after normalization?
pca_rows=[]

for outer_test in OUTER_TESTS:
    dev_mask=(df.test_id.to_numpy()!=outer_test)

    for method in ['RAW']+METHODS:
        X=TRANSFORMED[(method,'Spectral')][dev_mask]

        # Match the original NB02 convention:
        # StandardScaler is fitted on development observations only before PCA.
        scaler=StandardScaler().fit(X)
        Xz=scaler.transform(X)

        pca=PCA(n_components=min(5,Xz.shape[1]),random_state=SEED).fit(Xz)

        pca_rows.append({
            'outer_test':outer_test,
            'development_tests':','.join(map(str,[t for t in OUTER_TESTS if t!=outer_test])),
            'preprocessing':method,
            'pc1_explained_variance':float(pca.explained_variance_ratio_[0]),
            'pc2_explained_variance':float(pca.explained_variance_ratio_[1]),
            'pc1_pc2_cumulative':float(pca.explained_variance_ratio_[:2].sum()),
            'n_development':int(dev_mask.sum()),
        })

pca_diag=pd.DataFrame(pca_rows)
pca_diag.to_csv(TABLE_DIR/'02_pca_pc1_preprocessing_diagnostic.csv',index=False)
display(pca_diag.round(4))


## Model implementations

These reproduce the NB03/NB04 classification pipelines. The preprocessing sensitivity is applied **before** the original model-specific scaling/fitting logic.


In [ ]:
# 5. Metrics and model implementations
def classification_metrics(y_true,y_pred):
    y_true=np.asarray(y_true,dtype=int)
    y_pred=np.asarray(y_pred,dtype=int)
    observed=sorted(np.unique(y_true).tolist())
    ae=np.abs(y_true-y_pred)
    return {
        'accuracy':float(accuracy_score(y_true,y_pred)),
        'macro_f1_observed':float(f1_score(y_true,y_pred,labels=observed,average='macro',zero_division=0)),
        'macro_f1_fixed_1to5':float(f1_score(y_true,y_pred,labels=ALL_LABELS,average='macro',zero_division=0)),
        'balanced_accuracy':float(balanced_accuracy_score(y_true,y_pred)),
        'mcc':float(matthews_corrcoef(y_true,y_pred)),
        'qwk':float(cohen_kappa_score(y_true,y_pred,labels=ALL_LABELS,weights='quadratic')),
        'ordinal_mae':float(mean_absolute_error(y_true,y_pred)),
        'within_1_stage':float(np.mean(ae<=1)),
        'severe_error_ge2':float(np.mean(ae>=2)),
        'observed_labels':','.join(map(str,observed)),
    }

def fit_predict(model_name,params,X_train,y_train,X_eval,seed):
    y_train=np.asarray(y_train,dtype=int)

    if model_name=='OrdinalLogistic':
        scaler=StandardScaler().fit(X_train)
        Xt=scaler.transform(X_train)
        Xq=scaler.transform(X_eval)
        model=mord.LogisticAT(alpha=float(params['alpha']))
        model.fit(Xt,y_train)
        pred=model.predict(Xq).astype(int)

    elif model_name=='PLSDA':
        scaler=StandardScaler().fit(X_train)
        Xt=scaler.transform(X_train)
        Xq=scaler.transform(X_eval)
        classes=np.sort(np.unique(y_train))
        Y=np.zeros((len(y_train),len(classes)),dtype=float)
        for j,c in enumerate(classes):
            Y[:,j]=(y_train==c).astype(float)
        ncomp=int(min(params['n_components'],Xt.shape[1],Xt.shape[0]-1))
        model=PLSRegression(n_components=ncomp,scale=False,max_iter=1000)
        model.fit(Xt,Y)
        scores=np.asarray(model.predict(Xq))
        if scores.ndim==1:
            scores=scores[:,None]
        pred=classes[np.argmax(scores,axis=1)].astype(int)

    elif model_name=='SVM_RBF':
        scaler=StandardScaler().fit(X_train)
        Xt=scaler.transform(X_train)
        Xq=scaler.transform(X_eval)
        model=SVC(
            kernel='rbf',C=float(params['C']),gamma=params['gamma'],
            probability=False,cache_size=1800,random_state=seed
        )
        model.fit(Xt,y_train)
        pred=model.predict(Xq).astype(int)

    elif model_name=='RandomForest':
        model=RandomForestClassifier(
            n_estimators=int(params['n_estimators']),
            max_depth=params['max_depth'],
            min_samples_leaf=int(params['min_samples_leaf']),
            max_features=params['max_features'],
            random_state=seed,
            n_jobs=N_JOBS
        )
        model.fit(X_train,y_train)
        pred=model.predict(X_eval).astype(int)

    elif model_name=='XGBoost':
        classes=np.sort(np.unique(y_train))
        to_zero={c:i for i,c in enumerate(classes)}
        yz=np.array([to_zero[c] for c in y_train],dtype=int)
        model=xgb.XGBClassifier(
            objective='multi:softprob',
            num_class=len(classes),
            n_estimators=int(params['n_estimators']),
            max_depth=int(params['max_depth']),
            learning_rate=float(params['learning_rate']),
            subsample=float(params['subsample']),
            colsample_bytree=float(params['colsample_bytree']),
            reg_lambda=float(params.get('reg_lambda',1.0)),
            min_child_weight=float(params.get('min_child_weight',1.0)),
            tree_method='hist',
            eval_metric='mlogloss',
            random_state=seed,
            n_jobs=N_JOBS
        )
        model.fit(X_train,yz,verbose=False)
        pred=classes[np.asarray(model.predict(X_eval),dtype=int)].astype(int)

    else:
        raise ValueError(model_name)

    return np.clip(pred,1,5).astype(int)

def candidate_grid(model_name,modality):
    if model_name=='OrdinalLogistic':
        return [{'alpha':0.01},{'alpha':0.1},{'alpha':1.0},{'alpha':10.0}]
    if model_name=='PLSDA':
        return [{'n_components':c} for c in [2,4,8]]
    if model_name=='SVM_RBF':
        return [
            {'C':0.5,'gamma':'scale'},
            {'C':2.0,'gamma':'scale'},
            {'C':8.0,'gamma':'scale'},
            {'C':8.0,'gamma':0.03},
        ]
    if model_name=='RandomForest':
        return [
            {'n_estimators':400,'max_depth':None,'min_samples_leaf':1,'max_features':'sqrt'},
            {'n_estimators':400,'max_depth':18,'min_samples_leaf':1,'max_features':'sqrt'},
            {'n_estimators':400,'max_depth':None,'min_samples_leaf':3,'max_features':'sqrt'},
            {'n_estimators':400,'max_depth':None,'min_samples_leaf':1,'max_features':0.7},
        ]
    if model_name=='XGBoost':
        return [
            {'n_estimators':250,'max_depth':3,'learning_rate':0.05,'subsample':0.9,'colsample_bytree':0.9},
            {'n_estimators':350,'max_depth':5,'learning_rate':0.05,'subsample':0.9,'colsample_bytree':0.9},
            {'n_estimators':250,'max_depth':3,'learning_rate':0.10,'subsample':0.9,'colsample_bytree':0.9},
            {'n_estimators':500,'max_depth':2,'learning_rate':0.03,'subsample':0.9,'colsample_bytree':0.9},
        ]
    raise ValueError(model_name)

grid_rows=[]
for modality in MODALITY_NAMES:
    for model in MODELS:
        grid_rows.append({
            'modality':modality,
            'model':model,
            'candidate_count':len(candidate_grid(model,modality))
        })

grid_summary=pd.DataFrame(grid_rows)
grid_summary.to_csv(TABLE_DIR/'03_prespecified_grid_summary.csv',index=False)

assert grid_summary.groupby('modality').candidate_count.sum().eq(19).all()
print('Model functions and frozen compact grid ready.')
display(grid_summary)


## Nested inner LOEO under each preprocessing sensitivity

Expected fits:

- 2 preprocessing arms
- 2 modalities
- 4 outer folds
- 3 inner validation experiments
- 19 hyperparameter candidates per modality

Total: **912 inner fits**.


In [ ]:
# 6. Inner LOEO sensitivity
inner_rows=[]
leakage_rows=[]
fit_counter=0
inner_start=time.perf_counter()

test_id_arr=df.test_id.to_numpy(int)
y_all=df.stage.to_numpy(int)

for method_idx,method in enumerate(METHODS):
    for modality_idx,modality in enumerate(MODALITY_NAMES):
        Xall=TRANSFORMED[(method,modality)]

        for outer_test in OUTER_TESTS:
            dev_tests=[t for t in OUTER_TESTS if t!=outer_test]
            outer_mask=(test_id_arr==outer_test)

            print(f'\n{method} | {modality} | outer Test {outer_test} | dev={dev_tests}')

            for model_idx,model_name in enumerate(MODELS):
                candidates=candidate_grid(model_name,modality)

                for cand_idx,params in enumerate(candidates):
                    pjson=json.dumps(params,sort_keys=True)

                    for inner_val_test in dev_tests:
                        inner_train_tests=[t for t in dev_tests if t!=inner_val_test]
                        tr_mask=np.isin(test_id_arr,inner_train_tests)
                        va_mask=(test_id_arr==inner_val_test)

                        assert not np.any(tr_mask & va_mask)
                        assert not np.any(tr_mask & outer_mask)
                        assert not np.any(va_mask & outer_mask)

                        seed=(
                            SEED
                            + method_idx*10000
                            + modality_idx*5000
                            + outer_test*100
                            + inner_val_test*10
                            + cand_idx
                        )

                        t0=time.perf_counter()
                        status='ok'
                        err=''
                        try:
                            pred=fit_predict(
                                model_name,params,
                                Xall[tr_mask],y_all[tr_mask],
                                Xall[va_mask],seed
                            )
                            met=classification_metrics(y_all[va_mask],pred)
                        except Exception as e:
                            status='error'
                            err=repr(e)
                            met={k:np.nan for k in [
                                'accuracy','macro_f1_observed','macro_f1_fixed_1to5',
                                'balanced_accuracy','mcc','qwk','ordinal_mae',
                                'within_1_stage','severe_error_ge2'
                            ]}
                            met['observed_labels']=','.join(map(
                                str,sorted(np.unique(y_all[va_mask]).tolist())
                            ))

                        sec=time.perf_counter()-t0
                        fit_counter+=1

                        inner_rows.append({
                            'preprocessing':method,
                            'outer_test':outer_test,
                            'development_tests':','.join(map(str,dev_tests)),
                            'inner_val_test':inner_val_test,
                            'inner_train_tests':','.join(map(str,inner_train_tests)),
                            'modality':modality,
                            'n_features':Xall.shape[1],
                            'model':model_name,
                            'candidate_id':cand_idx,
                            'params_json':pjson,
                            'n_train':int(tr_mask.sum()),
                            'n_val':int(va_mask.sum()),
                            'status':status,
                            'error':err,
                            'fit_seconds':sec,
                            **met
                        })

                        leakage_rows.append({
                            'preprocessing':method,
                            'outer_test':outer_test,
                            'inner_val_test':inner_val_test,
                            'modality':modality,
                            'model':model_name,
                            'candidate_id':cand_idx,
                            'outer_rows_used_in_train':int(np.sum(tr_mask & outer_mask)),
                            'outer_rows_used_in_validation':int(np.sum(va_mask & outer_mask)),
                            'train_val_overlap_rows':int(np.sum(tr_mask & va_mask)),
                        })

                        if fit_counter % 100==0:
                            print(f'  {fit_counter}/912 inner fits | elapsed {(time.perf_counter()-inner_start)/60:.1f} min')

inner_results=pd.DataFrame(inner_rows)
inner_leakage=pd.DataFrame(leakage_rows)

inner_results.to_csv(TABLE_DIR/'04_inner_cv_fold_results_sensitivity.csv.gz',
                     index=False,compression='gzip')
inner_leakage.to_csv(TABLE_DIR/'05_inner_cv_leakage_audit.csv',index=False)

assert fit_counter==912,fit_counter
assert inner_results.status.eq('ok').all(),inner_results.loc[~inner_results.status.eq('ok'),['preprocessing','outer_test','modality','model','error']]
assert inner_leakage[['outer_rows_used_in_train','outer_rows_used_in_validation','train_val_overlap_rows']].to_numpy().sum()==0

print('\nInner LOEO complete.')
print('Leakage audit: PASS')
print('Elapsed minutes:',round((time.perf_counter()-inner_start)/60,2))


In [ ]:
# 7. Aggregate candidates and select hyperparameters using the frozen NB03 rule
ok=inner_results[inner_results.status.eq('ok')].copy()

group_cols=['preprocessing','outer_test','modality','model','candidate_id','params_json']
agg=ok.groupby(group_cols,as_index=False).agg(
    inner_folds=('inner_val_test','nunique'),
    mean_macro_f1_observed=('macro_f1_observed','mean'),
    sd_macro_f1_observed=('macro_f1_observed','std'),
    worst_macro_f1_observed=('macro_f1_observed','min'),
    mean_macro_f1_fixed_1to5=('macro_f1_fixed_1to5','mean'),
    mean_balanced_accuracy=('balanced_accuracy','mean'),
    mean_mcc=('mcc','mean'),
    mean_qwk=('qwk','mean'),
    mean_ordinal_mae=('ordinal_mae','mean'),
    mean_within_1_stage=('within_1_stage','mean'),
    mean_severe_error_ge2=('severe_error_ge2','mean'),
    total_fit_seconds=('fit_seconds','sum')
)
agg['eligible']=agg.inner_folds.eq(3)
agg.to_csv(TABLE_DIR/'06_inner_cv_candidate_summary_sensitivity.csv',index=False)

selected_rows=[]
for keys,g in agg[agg.eligible].groupby(['preprocessing','outer_test','modality','model'],sort=True):
    gs=g.sort_values(
        ['mean_macro_f1_observed','worst_macro_f1_observed','mean_qwk','mean_ordinal_mae','candidate_id'],
        ascending=[False,False,False,True,True],
        kind='mergesort'
    )
    selected_rows.append(gs.iloc[0].copy())

selected=pd.DataFrame(selected_rows).reset_index(drop=True)
assert len(selected)==2*4*2*5

selected['inner_model_rank']=selected.groupby(
    ['preprocessing','outer_test','modality']
)['mean_macro_f1_observed'].rank(method='min',ascending=False)

selected.to_csv(TABLE_DIR/'07_selected_hyperparameters_sensitivity.csv',index=False)

print('Selected hyperparameters:',len(selected))
display(selected.head(10))


In [ ]:
# 8. Outer LOEO for the two preprocessing arms
outer_rows=[]
outer_fit_counter=0
outer_start=time.perf_counter()

for method_idx,method in enumerate(METHODS):
    for modality_idx,modality in enumerate(MODALITY_NAMES):
        Xall=TRANSFORMED[(method,modality)]

        for outer_test in OUTER_TESTS:
            tr_mask=(test_id_arr!=outer_test)
            te_mask=(test_id_arr==outer_test)

            assert not np.any(tr_mask & te_mask)

            for model_idx,model_name in enumerate(MODELS):
                srow=selected[
                    selected.preprocessing.eq(method) &
                    selected.outer_test.eq(outer_test) &
                    selected.modality.eq(modality) &
                    selected.model.eq(model_name)
                ]
                assert len(srow)==1
                s=srow.iloc[0]
                params=json.loads(s.params_json)

                seed=(
                    SEED
                    + method_idx*10000
                    + modality_idx*5000
                    + outer_test*100
                    + model_idx*10
                )

                t0=time.perf_counter()
                status='ok'
                err=''
                try:
                    pred=fit_predict(
                        model_name,params,
                        Xall[tr_mask],y_all[tr_mask],
                        Xall[te_mask],seed
                    )
                    met=classification_metrics(y_all[te_mask],pred)
                except Exception as e:
                    status='error'
                    err=repr(e)
                    met={k:np.nan for k in [
                        'accuracy','macro_f1_observed','macro_f1_fixed_1to5',
                        'balanced_accuracy','mcc','qwk','ordinal_mae',
                        'within_1_stage','severe_error_ge2'
                    ]}
                    met['observed_labels']=','.join(map(
                        str,sorted(np.unique(y_all[te_mask]).tolist())
                    ))

                sec=time.perf_counter()-t0
                outer_fit_counter+=1

                outer_rows.append({
                    'preprocessing':method,
                    'outer_test':outer_test,
                    'development_tests':','.join(map(str,[t for t in OUTER_TESTS if t!=outer_test])),
                    'modality':modality,
                    'model':model_name,
                    'candidate_id':int(s.candidate_id),
                    'params_json':s.params_json,
                    'inner_mean_macro_f1':float(s.mean_macro_f1_observed),
                    'inner_worst_macro_f1':float(s.worst_macro_f1_observed),
                    'inner_model_rank':float(s.inner_model_rank),
                    'n_train':int(tr_mask.sum()),
                    'n_test':int(te_mask.sum()),
                    'status':status,
                    'error':err,
                    'fit_seconds':sec,
                    **met
                })

                print(
                    f'{method:8s} | {modality:12s} | T{outer_test} | '
                    f'{model_name:15s} | Macro-F1={met["macro_f1_observed"]:.4f}'
                )

outer_sens=pd.DataFrame(outer_rows)
outer_sens.to_csv(TABLE_DIR/'08_outer_loeo_metrics_sensitivity.csv',index=False)

assert outer_fit_counter==80
assert outer_sens.status.eq('ok').all(),outer_sens.loc[~outer_sens.status.eq('ok'),['preprocessing','outer_test','modality','model','error']]

print('\nOuter sensitivity complete.')
print('Elapsed minutes:',round((time.perf_counter()-outer_start)/60,2))


In [ ]:
# 9. Load frozen RAW baseline from the canonical NB04 run
LATEST_NB04=RESULTS_ROOT/'LATEST_NB04_RUN.json'
assert LATEST_NB04.exists(),LATEST_NB04

nb04_ptr=json.loads(LATEST_NB04.read_text(encoding='utf-8'))
NB04_RUN=Path(nb04_ptr['run_dir'])

RAW_METRICS_PATH=NB04_RUN/'tables'/'outer_loeo_metrics_all_models.csv'
RAW_RANK_PATH=NB04_RUN/'tables'/'inner_vs_outer_model_ranking.csv'
assert RAW_METRICS_PATH.exists(),RAW_METRICS_PATH
assert RAW_RANK_PATH.exists(),RAW_RANK_PATH

raw_metrics=pd.read_csv(RAW_METRICS_PATH)
raw_rank=pd.read_csv(RAW_RANK_PATH)

raw_metrics=raw_metrics[raw_metrics.modality.isin(MODALITY_NAMES)].copy()
raw_rank=raw_rank[raw_rank.modality.isin(MODALITY_NAMES)].copy()

raw_metrics.insert(0,'preprocessing','RAW')

assert len(raw_metrics)==4*2*5
assert len(raw_rank)==4*2*5

baseline_hashes=pd.DataFrame([
    {'artifact':str(RAW_METRICS_PATH.relative_to(PROJECT)),
     'sha256':sha256_file(RAW_METRICS_PATH),'bytes':RAW_METRICS_PATH.stat().st_size},
    {'artifact':str(RAW_RANK_PATH.relative_to(PROJECT)),
     'sha256':sha256_file(RAW_RANK_PATH),'bytes':RAW_RANK_PATH.stat().st_size},
])
baseline_hashes.to_csv(META_DIR/'raw_nb04_source_hashes.csv',index=False)

print('RAW baseline:',NB04_RUN.name)
print('Frozen raw baseline integrity recorded.')


In [ ]:
# 10. Internally selected model under each condition

# RAW: use the exact NB04 internal rank.
raw_selected=raw_rank[raw_rank.inner_model_rank.eq(1)].copy()
assert len(raw_selected)==8
raw_selected=raw_selected[[
    'outer_test','modality','model',
    'inner_mean_macro_f1_rankfile','macro_f1_observed',
    'balanced_accuracy','mcc','qwk','ordinal_mae'
]].rename(columns={
    'inner_mean_macro_f1_rankfile':'inner_mean_macro_f1'
})
raw_selected.insert(0,'preprocessing','RAW')

# Sensitivity arms: preserve the internal-selection rule.
sens_selected_rows=[]
for keys,g in outer_sens.groupby(['preprocessing','outer_test','modality'],sort=True):
    gs=g.sort_values(
        ['inner_mean_macro_f1','inner_worst_macro_f1','model'],
        ascending=[False,False,True],
        kind='mergesort'
    )
    sens_selected_rows.append(gs.iloc[0].copy())

sens_selected=pd.DataFrame(sens_selected_rows)[[
    'preprocessing','outer_test','modality','model',
    'inner_mean_macro_f1','macro_f1_observed',
    'balanced_accuracy','mcc','qwk','ordinal_mae'
]]

selected_all=pd.concat([raw_selected,sens_selected],ignore_index=True)
assert len(selected_all)==24

# Uniform random reference for each external experiment.
ref_rows=[]
for t in OUTER_TESTS:
    te=df[df.test_id.eq(t)]
    p=np.array([(te.stage==k).mean() for k in ALL_LABELS],float)
    q=np.repeat(0.2,5)
    f1=np.where((p+q)>0,2*p*q/(p+q),0.0)
    obs=sorted(te.stage.unique().tolist())
    idx=[ALL_LABELS.index(int(c)) for c in obs]
    ref_rows.append({
        'outer_test':t,
        'uniform_random_macro_f1_observed':float(f1[idx].mean())
    })
random_ref=pd.DataFrame(ref_rows)

selected_all=selected_all.merge(random_ref,on='outer_test',how='left')
selected_all['above_uniform_random']=(
    selected_all.macro_f1_observed-selected_all.uniform_random_macro_f1_observed
)

selected_all.to_csv(TABLE_DIR/'09_selected_model_performance_raw_snv_domainz.csv',index=False)
display(selected_all.round(4))


In [ ]:
# 11. Delta relative to frozen RAW selected model
raw_base=selected_all[selected_all.preprocessing.eq('RAW')][[
    'outer_test','modality','model','macro_f1_observed',
    'balanced_accuracy','mcc','qwk','ordinal_mae'
]].rename(columns={
    'model':'raw_selected_model',
    'macro_f1_observed':'raw_macro_f1',
    'balanced_accuracy':'raw_balanced_accuracy',
    'mcc':'raw_mcc',
    'qwk':'raw_qwk',
    'ordinal_mae':'raw_ordinal_mae'
})

sens_compare=selected_all[~selected_all.preprocessing.eq('RAW')].merge(
    raw_base,on=['outer_test','modality'],how='left',validate='many_to_one'
)

sens_compare['delta_macro_f1_vs_raw']=sens_compare.macro_f1_observed-sens_compare.raw_macro_f1
sens_compare['delta_balanced_accuracy_vs_raw']=sens_compare.balanced_accuracy-sens_compare.raw_balanced_accuracy
sens_compare['delta_mcc_vs_raw']=sens_compare.mcc-sens_compare.raw_mcc
sens_compare['delta_qwk_vs_raw']=sens_compare.qwk-sens_compare.raw_qwk
sens_compare['delta_ordinal_mae_vs_raw']=sens_compare.ordinal_mae-sens_compare.raw_ordinal_mae

sens_compare.to_csv(TABLE_DIR/'10_selected_model_delta_vs_raw.csv',index=False)

summary=sens_compare.groupby(['preprocessing','modality'],as_index=False).agg(
    mean_raw_macro_f1=('raw_macro_f1','mean'),
    mean_sensitivity_macro_f1=('macro_f1_observed','mean'),
    mean_delta_macro_f1=('delta_macro_f1_vs_raw','mean'),
    median_delta_macro_f1=('delta_macro_f1_vs_raw','median'),
    min_delta_macro_f1=('delta_macro_f1_vs_raw','min'),
    max_delta_macro_f1=('delta_macro_f1_vs_raw','max'),
    scenarios_improved=('delta_macro_f1_vs_raw',lambda x:int(np.sum(np.asarray(x)>0))),
    scenarios_total=('delta_macro_f1_vs_raw','size'),
    mean_delta_balanced_accuracy=('delta_balanced_accuracy_vs_raw','mean'),
    mean_delta_mcc=('delta_mcc_vs_raw','mean'),
    mean_delta_qwk=('delta_qwk_vs_raw','mean'),
    mean_delta_ordinal_mae=('delta_ordinal_mae_vs_raw','mean'),
)
summary.to_csv(TABLE_DIR/'11_preprocessing_effect_summary.csv',index=False)
display(summary.round(4))


In [ ]:
# 12. Figure A — selected external Macro-F1 across preprocessing conditions
fig,axes=plt.subplots(1,2,figsize=(12,4.8),sharey=True)

for ax,modality in zip(axes,MODALITY_NAMES):
    z=selected_all[selected_all.modality.eq(modality)].copy()
    for method,marker in [('RAW','o'),('SNV','s'),('DOMAIN_Z','^')]:
        g=z[z.preprocessing.eq(method)].sort_values('outer_test')
        ax.plot(
            g.outer_test,g.macro_f1_observed,
            marker=marker,linewidth=2,label=method
        )

    rr=random_ref.sort_values('outer_test')
    ax.plot(
        rr.outer_test,rr.uniform_random_macro_f1_observed,
        linestyle='--',marker='x',label='Uniform random'
    )

    ax.set_xticks(OUTER_TESTS)
    ax.set_xlabel('Held-out experiment')
    ax.set_title(modality)
    ax.grid(alpha=0.25)

axes[0].set_ylabel('External Macro-F1')
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='lower center',ncol=4,bbox_to_anchor=(0.5,-0.04))
fig.suptitle('Selected-model performance after spectral normalization sensitivity',y=1.02)
fig.tight_layout(rect=[0,0.07,1,1])

for ext in ['png','pdf','svg']:
    fig.savefig(
        FIG_DIR/f'Figure_R2A_selected_macroF1_preprocessing.{ext}',
        dpi=300 if ext=='png' else None,
        bbox_inches='tight'
    )
plt.close(fig)

# Figure B — PC1 explained variance
fig,ax=plt.subplots(figsize=(8.5,5.2))
for method,marker in [('RAW','o'),('SNV','s'),('DOMAIN_Z','^')]:
    g=pca_diag[pca_diag.preprocessing.eq(method)].sort_values('outer_test')
    ax.plot(g.outer_test,g.pc1_explained_variance,marker=marker,linewidth=2,label=method)

ax.set_xticks(OUTER_TESTS)
ax.set_xlabel('Held-out experiment')
ax.set_ylabel('PC1 explained variance ratio')
ax.set_title('Development-set spectral PC1 dominance after preprocessing')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()

for ext in ['png','pdf','svg']:
    fig.savefig(
        FIG_DIR/f'Figure_R2B_PC1_variance_preprocessing.{ext}',
        dpi=300 if ext=='png' else None,
        bbox_inches='tight'
    )
plt.close(fig)

# Figure C — delta heatmap-like table
plot=sens_compare.pivot_table(
    index=['preprocessing','modality'],
    columns='outer_test',
    values='delta_macro_f1_vs_raw'
).sort_index()

fig,ax=plt.subplots(figsize=(7.5,4.0))
im=ax.imshow(plot.to_numpy(),aspect='auto')
ax.set_xticks(range(len(plot.columns)),[f'Test {c}' for c in plot.columns])
ax.set_yticks(
    range(len(plot.index)),
    [f'{a} | {b}' for a,b in plot.index]
)
ax.set_title('Change in selected-model Macro-F1 relative to frozen RAW baseline')
for i in range(plot.shape[0]):
    for j in range(plot.shape[1]):
        ax.text(j,i,f'{plot.iloc[i,j]:+.3f}',ha='center',va='center')
fig.colorbar(im,ax=ax,label='Δ Macro-F1 vs RAW')
fig.tight_layout()

for ext in ['png','pdf','svg']:
    fig.savefig(
        FIG_DIR/f'Figure_R2C_delta_macroF1_vs_raw.{ext}',
        dpi=300 if ext=='png' else None,
        bbox_inches='tight'
    )
plt.close(fig)


In [ ]:
# 13. Machine-readable key results and manuscript notes
key={}

for method in METHODS:
    g=sens_compare[sens_compare.preprocessing.eq(method)]
    key[method]={
        'mean_delta_macro_f1_all_8_scenarios':float(g.delta_macro_f1_vs_raw.mean()),
        'median_delta_macro_f1_all_8_scenarios':float(g.delta_macro_f1_vs_raw.median()),
        'n_improved_of_8':int((g.delta_macro_f1_vs_raw>0).sum()),
        'max_improvement':float(g.delta_macro_f1_vs_raw.max()),
        'max_degradation':float(g.delta_macro_f1_vs_raw.min()),
        'mean_macro_f1':float(g.macro_f1_observed.mean()),
    }

key['RAW']={
    'mean_selected_macro_f1_8_scenarios':float(
        selected_all[selected_all.preprocessing.eq('RAW')].macro_f1_observed.mean()
    )
}

pca_key=pca_diag.groupby('preprocessing').pc1_explained_variance.agg(['mean','min','max']).reset_index()
key['PC1']={
    r.preprocessing:{
        'mean':float(r['mean']),
        'min':float(r['min']),
        'max':float(r['max'])
    }
    for _,r in pca_key.iterrows()
}

(TABLE_DIR/'12_key_results_reviewer2.json').write_text(
    json.dumps(key,indent=2,ensure_ascii=False),
    encoding='utf-8'
)

notes=[
    '# Reviewer 2 — spectral normalization sensitivity',
    '',
    '## Design',
    '- RAW values are imported unchanged from frozen NB04.',
    '- SNV is applied row-wise across the 18 spectral bands.',
    '- DOMAIN_Z standardizes each experiment using that experiment’s unlabeled feature distribution.',
    '- For DOMAIN_Z, no ripeness labels from the target experiment are used. It is an explicitly transductive diagnostic.',
    '- Hyperparameters and model family are re-selected by the original inner LOEO rule under each transformed representation.',
    '',
    '## Selected-model Macro-F1 changes vs RAW',
]

for method in METHODS:
    g=sens_compare[sens_compare.preprocessing.eq(method)]
    notes += [
        f'### {method}',
        f'- Mean Δ Macro-F1 across 8 outer-test × modality scenarios: {g.delta_macro_f1_vs_raw.mean():+.4f}',
        f'- Median Δ Macro-F1: {g.delta_macro_f1_vs_raw.median():+.4f}',
        f'- Improved scenarios: {(g.delta_macro_f1_vs_raw>0).sum()}/8',
        f'- Largest improvement: {g.delta_macro_f1_vs_raw.max():+.4f}',
        f'- Largest degradation: {g.delta_macro_f1_vs_raw.min():+.4f}',
        ''
    ]

notes += [
    '## PC1 diagnostic',
]
for method in ['RAW']+METHODS:
    g=pca_diag[pca_diag.preprocessing.eq(method)]
    notes.append(
        f'- {method}: mean development-set PC1 explained variance = {g.pc1_explained_variance.mean():.4f} '
        f'(range {g.pc1_explained_variance.min():.4f}–{g.pc1_explained_variance.max():.4f}).'
    )

notes += [
    '',
    '## Interpretation rule',
    'If SNV or DOMAIN_Z materially restores external performance across several held-out experiments, '
    'the manuscript must state that a substantial part of the apparent experiment shift is compatible '
    'with correctable intensity/location-scale effects. If PC1 dominance collapses but predictive '
    'transportability remains poor, this supports the narrower conclusion that simple additive/'
    'multiplicative normalization is insufficient to restore the sensor–ripeness relationship.',
    '',
    'Do not describe DOMAIN_Z as ordinary inductive deployment. It uses unlabeled target-batch statistics.',
]

(TEXT_DIR/'reviewer2_spectral_normalization_notes.md').write_text(
    '\n'.join(notes)+'\n',encoding='utf-8'
)
print('\n'.join(notes))


In [ ]:
# 14. Manifest, package hashes, ZIP and browser download
versions={
    'python':platform.python_version(),
    'numpy':np.__version__,
    'pandas':pd.__version__,
    'matplotlib':plt.matplotlib.__version__,
    'scikit_learn':sklearn.__version__,
    'mord':getattr(mord,'__version__','0.7/unknown'),
    'xgboost':xgb.__version__,
}

elapsed=time.perf_counter()-start_all

manifest={
    'project':'BANANA_RIPENESS_MULTISENSOR_Q1',
    'notebook':'10_REVIEWER2_SPECTRAL_NORMALIZATION_SENSITIVITY',
    'run_id':RUN_ID,
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'seed':SEED,
    'source_master':str(MASTER_PATH),
    'source_protocol':str(PROTOCOL_PATH),
    'source_hashes':source_hashes.to_dict('records'),
    'raw_baseline_run':str(NB04_RUN),
    'methods':{
        'SNV':'row-wise standard normal variate on 18 spectral bands',
        'DOMAIN_Z':'per-experiment unlabeled feature z-score; transductive diagnostic'
    },
    'modalities':MODALITIES,
    'models':MODELS,
    'selection_rule':[
        'maximize mean observed-label Macro-F1 across 3 inner experiment folds',
        'maximize worst inner-fold observed-label Macro-F1',
        'maximize mean QWK',
        'minimize mean ordinal MAE'
    ],
    'inner_fits':int(fit_counter),
    'outer_fits':int(outer_fit_counter),
    'outer_label_use_for_preprocessing':False,
    'domain_z_target_feature_use':'yes; unlabeled target-batch features only',
    'elapsed_minutes':elapsed/60,
    'versions':versions,
    'key_results':key
}

(META_DIR/'run_manifest.json').write_text(
    json.dumps(manifest,indent=2,ensure_ascii=False),
    encoding='utf-8'
)

(META_DIR/'environment.txt').write_text(
    '\n'.join([
        f'platform={platform.platform()}',
        f'cpu_count={os.cpu_count()}',
        *[f'{k}={v}' for k,v in versions.items()]
    ])+'\n',
    encoding='utf-8'
)

readme=[
    '# NB10 Reviewer 2 — spectral normalization sensitivity',
    '',
    f'Run ID: `{RUN_ID}`',
    f'Runtime: {elapsed/60:.2f} min',
    '',
    '- RAW = frozen NB04 reference; not recomputed.',
    '- SNV = row-wise spectral normalization.',
    '- DOMAIN_Z = per-experiment unlabeled z-score diagnostic.',
    '- Full nested inner LOEO model/hyperparameter selection repeated for both sensitivity arms.',
    '- No target labels used in preprocessing or selection.',
    '',
    'Send the ZIP generated by this notebook back to ChatGPT for scientific interpretation.'
]
(RUN_DIR/'README_RESULTS.md').write_text('\n'.join(readme)+'\n',encoding='utf-8')

hash_rows=[]
for p in sorted(RUN_DIR.rglob('*')):
    if p.is_file() and p.name!='package_hashes.csv':
        hash_rows.append({
            'relative_path':str(p.relative_to(RUN_DIR)),
            'bytes':p.stat().st_size,
            'sha256':sha256_file(p)
        })
pd.DataFrame(hash_rows).to_csv(META_DIR/'package_hashes.csv',index=False)

zip_base=RESULTS_ROOT/f'RESULTS_NB10_REVIEWER2_SPECTRAL_NORMALIZATION_{RUN_ID}'
zip_path=Path(shutil.make_archive(
    str(zip_base),'zip',
    root_dir=RUN_DIR.parent,
    base_dir=RUN_DIR.name
))

latest={
    'notebook':'10_REVIEWER2_SPECTRAL_NORMALIZATION_SENSITIVITY',
    'run_id':RUN_ID,
    'run_dir':str(RUN_DIR),
    'zip_path':str(zip_path),
    'zip_bytes':zip_path.stat().st_size,
    'elapsed_minutes':elapsed/60
}
(RESULTS_ROOT/'LATEST_NB10_REVIEWER2_RUN.json').write_text(
    json.dumps(latest,indent=2),
    encoding='utf-8'
)

print('\n✅ NB10 COMPLETED')
print('Run directory:',RUN_DIR)
print('ZIP:',zip_path)
print('ZIP size (MB):',round(zip_path.stat().st_size/1024**2,2))
print('Elapsed minutes:',round(elapsed/60,2))
print('\nUpload this ZIP to ChatGPT.')

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Automatic browser download did not complete:',repr(e))
    print('ZIP is safely stored in Drive:',zip_path)
